# Sprint 3 — 11_model_comparison.ipynb
## Rol: Model Comparator

### Objetivo
Consolidar los resultados generados por `10_evaluation.ipynb`, rankear los modelos baseline por la métrica principal del proyecto y seleccionar los **top 2–3 candidatos** que pasarán al Sprint 4 para tuning.

### Qué sí hace este notebook
- Carga los outputs de evaluación del rol anterior.
- Consolida métricas CV y OOF en una tabla comparativa única.
- Rankea modelos según la métrica principal definida por el equipo.
- Analiza trade-offs entre desempeño, velocidad e interpretabilidad.
- Integra, si existe, el análisis complementario de train-vs-test gap.
- Propone y documenta los **top candidatos** para Sprint 4.
- Genera outputs exportables para el Sprint Review.

### Qué NO hace este notebook
- No reentrena modelos.
- No tunea hiperparámetros.
- No usa el test final para validación definitiva.
- No aplica stacking ni ensambles avanzados todavía.
- No modifica los `.pkl` baseline del rol anterior.

### Inputs esperados
**Obligatorios**
- `models/evaluation_cv_summary.csv`
- `models/evaluation_cv_fold_results.csv`
- `models/evaluation_oof_metrics.csv`
- `models/evaluation_classification_reports.csv`

**Opcionales**
- `models/evaluation_train_test_gap_subset.csv`
- `reports/figures/sprint3/roc_baselines.png`
- `reports/figures/sprint3/pr_baselines.png`
- `reports/figures/sprint3/confusion_matrices_baselines.png`

### Outputs esperados
- `models/model_comparison_summary.csv`
- `models/model_selection_candidates.csv`
- `models/model_discard_reasons.csv`
- `reports/figures/sprint3/model_comparison_primary_metric.png`
- `reports/figures/sprint3/model_comparison_tradeoffs.png`

### Nota metodológica
La selección final debe estar alineada con la métrica principal definida en Sprint 1. Por defecto este notebook usa **F1** como criterio principal, pero puedes ajustar esa decisión al inicio si el equipo definió otra prioridad (por ejemplo Recall o AUC-ROC).

## 1. Checklist antes de correr
Antes de ejecutar este notebook, valida:
1. Estar en la rama correcta del rol (`feat/PB-13` o equivalente).
2. Haber traído de `main` lo mergeado en PB-12.
3. Tener los CSV de evaluación ya generados.
4. Tener claro cuál es la métrica principal del proyecto.
5. Recordar que los `.pkl` baseline ya están bajo DVC desde PB-11 y aquí no deben tocarse.

> Si falta `evaluation_train_test_gap_subset.csv`, el notebook igual debe correr. Ese archivo se tratará como opcional.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 2. Configuración del proyecto
Ajusta solo si tu estructura cambió.

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path(".").resolve()

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures" / "sprint3"

CV_SUMMARY_PATH = MODELS_DIR / "evaluation_cv_summary.csv"
CV_FOLD_PATH = MODELS_DIR / "evaluation_cv_fold_results.csv"
OOF_METRICS_PATH = MODELS_DIR / "evaluation_oof_metrics.csv"
CLASS_REPORTS_PATH = MODELS_DIR / "evaluation_classification_reports.csv"
GAP_SUBSET_PATH = MODELS_DIR / "evaluation_train_test_gap_subset.csv"

PRIMARY_METRIC_COLUMN = "test_f1_mean"   # ajustar si el equipo definió otra métrica principal
PRIMARY_METRIC_LABEL = "F1 (CV)"
TOP_K = 3                                # típicamente 2 o 3 según el PDF

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
required_files = [
    CV_SUMMARY_PATH,
    CV_FOLD_PATH,
    OOF_METRICS_PATH,
    CLASS_REPORTS_PATH,
]

optional_files = [
    GAP_SUBSET_PATH,
    REPORTS_DIR / "roc_baselines.png",
    REPORTS_DIR / "pr_baselines.png",
    REPORTS_DIR / "confusion_matrices_baselines.png",
]

print("=== Required files ===")
for path in required_files:
    print(path, "->", path.exists())

if not all(path.exists() for path in required_files):
    missing = [str(path) for path in required_files if not path.exists()]
    raise FileNotFoundError(f"Faltan inputs obligatorios para Model Comparator: {missing}")

print("\n=== Optional files ===")
for path in optional_files:
    print(path, "->", path.exists())

## 3. Carga de outputs del rol anterior

In [ ]:
cv_summary_df = pd.read_csv(CV_SUMMARY_PATH)
cv_fold_df = pd.read_csv(CV_FOLD_PATH)
oof_metrics_df = pd.read_csv(OOF_METRICS_PATH)
class_reports_df = pd.read_csv(CLASS_REPORTS_PATH)

if GAP_SUBSET_PATH.exists():
    gap_subset_df = pd.read_csv(GAP_SUBSET_PATH)
else:
    gap_subset_df = None

print("cv_summary_df:", cv_summary_df.shape)
print("cv_fold_df:", cv_fold_df.shape)
print("oof_metrics_df:", oof_metrics_df.shape)
print("class_reports_df:", class_reports_df.shape)
if gap_subset_df is not None:
    print("gap_subset_df:", gap_subset_df.shape)

In [ ]:
display(cv_summary_df)
display(oof_metrics_df)
if gap_subset_df is not None:
    display(gap_subset_df)

## 4. Validaciones mínimas
Aquí confirmamos que la métrica principal exista y que las tablas tengan la llave `model_key`.

In [ ]:
required_key = "model_key"

for df_name, df in {
    "cv_summary_df": cv_summary_df,
    "oof_metrics_df": oof_metrics_df,
    "class_reports_df": class_reports_df,
}.items():
    if required_key not in df.columns:
        raise KeyError(f"{df_name} no contiene la columna '{required_key}'.")

if PRIMARY_METRIC_COLUMN not in cv_summary_df.columns:
    raise KeyError(
        f"La métrica principal '{PRIMARY_METRIC_COLUMN}' no existe en cv_summary_df. "
        "Ajusta PRIMARY_METRIC_COLUMN al inicio del notebook."
    )

## 5. Construcción de tabla comparativa única
Se combinan:
- métricas CV (promedios y desviaciones),
- métricas OOF,
- y, si existe, el análisis de train-vs-test gap.

In [ ]:
comparison_df = cv_summary_df.merge(
    oof_metrics_df,
    on="model_key",
    how="left",
    validate="one_to_one",
)

if gap_subset_df is not None:
    comparison_df = comparison_df.merge(
        gap_subset_df,
        on="model_key",
        how="left",
    )

comparison_df = comparison_df.copy()
comparison_df["rank_primary_metric"] = comparison_df[PRIMARY_METRIC_COLUMN].rank(
    ascending=False, method="dense"
).astype(int)

comparison_df = comparison_df.sort_values(
    by=[PRIMARY_METRIC_COLUMN, "oof_f1", "test_roc_auc_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

comparison_df.insert(0, "rank", np.arange(1, len(comparison_df) + 1))
display(comparison_df)

## 6. Enriquecimiento cualitativo: interpretabilidad, velocidad y trade-offs
Esta sección no reemplaza la métrica principal, pero ayuda a justificar por qué un modelo pasa o se descarta.

In [ ]:
interpretability_map = {
    "lr": "alta",
    "dt": "alta",
    "rf": "media",
    "knn": "media-baja",
    "svm": "baja",
}

family_map = {
    "lr": "lineal",
    "dt": "árbol simple",
    "rf": "ensamble de árboles",
    "knn": "instancias / distancia",
    "svm": "margen máximo",
}

comparison_df["interpretability"] = comparison_df["model_key"].map(interpretability_map).fillna("no clasificada")
comparison_df["model_family"] = comparison_df["model_key"].map(family_map).fillna("otra")

if "fit_time_mean" in comparison_df.columns:
    slow_threshold = comparison_df["fit_time_mean"].median()
    comparison_df["speed_bucket"] = np.where(
        comparison_df["fit_time_mean"] <= slow_threshold,
        "más rápido",
        "más lento",
    )
else:
    comparison_df["speed_bucket"] = "sin dato"

if {"test_precision_mean", "test_recall_mean"}.issubset(comparison_df.columns):
    comparison_df["precision_recall_gap"] = (
        comparison_df["test_precision_mean"] - comparison_df["test_recall_mean"]
    )

display(comparison_df)

## 7. Selección de top candidatos para Sprint 4
Se propone una primera selección automática basada en la métrica principal y luego se documenta la justificación.
Ajusta manualmente la selección si el equipo decide otra combinación.

In [ ]:
comparison_df["selected_for_sprint4"] = False
top_indices = comparison_df.head(TOP_K).index
comparison_df.loc[top_indices, "selected_for_sprint4"] = True

def build_selection_reason(row):
    reasons = [f"Top {int(row['rank'])} por {PRIMARY_METRIC_LABEL}"]
    if "oof_f1" in row and pd.notna(row["oof_f1"]):
        reasons.append(f"OOF F1={row['oof_f1']:.3f}")
    if "test_roc_auc_mean" in row and pd.notna(row["test_roc_auc_mean"]):
        reasons.append(f"AUC CV={row['test_roc_auc_mean']:.3f}")
    reasons.append(f"Interpretabilidad: {row['interpretability']}")
    if "speed_bucket" in row:
        reasons.append(f"Velocidad: {row['speed_bucket']}")
    return " | ".join(reasons)

def build_discard_reason(row):
    reasons = []
    reasons.append(f"No quedó en top {TOP_K} por {PRIMARY_METRIC_LABEL}")
    if "speed_bucket" in row and row["speed_bucket"] == "más lento":
        reasons.append("mayor costo computacional")
    if "interpretability" in row and row["interpretability"] in {"baja", "media-baja"}:
        reasons.append("menor interpretabilidad")
    if "f1_gap" in row and pd.notna(row["f1_gap"]) and row["f1_gap"] >= 0.10:
        reasons.append("señal de posible overfitting")
    return " | ".join(reasons)

comparison_df["selection_reason"] = np.where(
    comparison_df["selected_for_sprint4"],
    comparison_df.apply(build_selection_reason, axis=1),
    "",
)

comparison_df["discard_reason"] = np.where(
    ~comparison_df["selected_for_sprint4"],
    comparison_df.apply(build_discard_reason, axis=1),
    "",
)

selected_df = comparison_df[comparison_df["selected_for_sprint4"]].copy()
discarded_df = comparison_df[~comparison_df["selected_for_sprint4"]].copy()

display(selected_df)
display(discarded_df)

## 8. Tabla resumen para Sprint Review
Esta tabla resume lo mínimo que debería verse en la review:
- ranking
- métrica principal
- métricas complementarias
- selección para Sprint 4

In [ ]:
sprint_review_cols = [
    "rank",
    "model_key",
    "model_family",
    "interpretability",
    "speed_bucket",
    PRIMARY_METRIC_COLUMN,
    "test_precision_mean",
    "test_recall_mean",
    "test_roc_auc_mean",
    "oof_f1",
    "oof_roc_auc",
    "selected_for_sprint4",
]

existing_review_cols = [c for c in sprint_review_cols if c in comparison_df.columns]
sprint_review_df = comparison_df[existing_review_cols].copy()

display(sprint_review_df)

## 9. Visualizaciones comparativas
Se generan dos figuras:
- comparación por métrica principal
- trade-off entre desempeño y tiempo de fit

In [ ]:
# Figura 1: métrica principal por modelo
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(comparison_df["model_key"], comparison_df[PRIMARY_METRIC_COLUMN])
ax.set_title(f"Comparación de modelos por {PRIMARY_METRIC_LABEL}")
ax.set_xlabel("Modelo")
ax.set_ylabel(PRIMARY_METRIC_LABEL)
plt.tight_layout()

primary_metric_fig_path = REPORTS_DIR / "model_comparison_primary_metric.png"
plt.savefig(primary_metric_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", primary_metric_fig_path)

In [ ]:
# Figura 2: trade-off desempeño vs tiempo
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(comparison_df["fit_time_mean"], comparison_df[PRIMARY_METRIC_COLUMN])

for _, row in comparison_df.iterrows():
    ax.annotate(row["model_key"], (row["fit_time_mean"], row[PRIMARY_METRIC_COLUMN]))

ax.set_title(f"Trade-off: tiempo de fit vs {PRIMARY_METRIC_LABEL}")
ax.set_xlabel("Fit time mean")
ax.set_ylabel(PRIMARY_METRIC_LABEL)
plt.tight_layout()

tradeoff_fig_path = REPORTS_DIR / "model_comparison_tradeoffs.png"
plt.savefig(tradeoff_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", tradeoff_fig_path)

## 10. Persistencia de outputs finales
Estos son los archivos que debe consumir o revisar el siguiente sprint / review.

In [ ]:
comparison_summary_path = MODELS_DIR / "model_comparison_summary.csv"
selection_candidates_path = MODELS_DIR / "model_selection_candidates.csv"
discard_reasons_path = MODELS_DIR / "model_discard_reasons.csv"

comparison_df.to_csv(comparison_summary_path, index=False)
selected_df.to_csv(selection_candidates_path, index=False)
discarded_df.to_csv(discard_reasons_path, index=False)

print("Saved:", comparison_summary_path)
print("Saved:", selection_candidates_path)
print("Saved:", discard_reasons_path)

## 11. Revisión de artefactos generados

In [ ]:
generated_paths = [
    comparison_summary_path,
    selection_candidates_path,
    discard_reasons_path,
    primary_metric_fig_path,
    tradeoff_fig_path,
]

generated_df = pd.DataFrame(
    {
        "artifact_path": [str(p) for p in generated_paths],
        "exists": [p.exists() for p in generated_paths],
        "size_kb": [round(p.stat().st_size / 1024, 2) if p.exists() else None for p in generated_paths],
    }
)

display(generated_df)

## 12. Handoff al siguiente sprint / review
### Lo que ya queda listo
- tabla comparativa consolidada
- ranking por métrica principal
- selección de top candidatos
- justificación de descarte
- figuras para Sprint Review

### Lo que sigue después
En Sprint 4 se debe:
- tomar los candidatos seleccionados,
- hacer tuning de hiperparámetros,
- probar modelos avanzados si corresponde,
- y comparar contra estos baselines ya priorizados.

### Notas de versionado
- Estos outputs (`.csv`, `.png`) normalmente son livianos y deberían quedar en Git.
- Los `.pkl` baseline del Sprint 3 siguen bajo DVC desde PB-11.
- Si algún CSV/PNG creciera demasiado, se puede mover a DVC, pero no es el caso típico para esta etapa.
- No agregar al PR archivos `*_partial.csv` si solo fueron checkpoints de trabajo.